<a href="https://colab.research.google.com/github/ShumwayRobert1980/AI-Prompt-Genius/blob/main/Colab-TextGen-GPU.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
A

# oobabooga/textgen

After running both cells, a public gradio URL will appear at the bottom in around 10 minutes. You can optionally generate an API link.

* Project page: https://github.com/oobabooga/textgen
* Gradio server status: https://status.gradio.app/

In [ ]:
#@title 1. Keep this tab alive to prevent Colab from disconnecting you { display-mode: "form" }

#@markdown Press play on the music player that will appear below:
%%html
<audio src="https://oobabooga.github.io/silence.m4a" controls>

In [ ]:
A

In [ ]:
import os
from pathlib import Path

os.environ.pop('PYTHONPATH', None)
os.environ.pop('MPLBACKEND', None)

if Path.cwd().name != 'textgen':
  print("\033[1;32;1m\n --> Installing the web UI. This will take a while, but after the initial setup, you can download and test as many models as you like.\033[0;37;0m\n")

  !git clone https://github.com/oobabooga/textgen
  %cd textgen

  # Install the project in an isolated environment
  !GPU_CHOICE=A \
  LAUNCH_AFTER_INSTALL=FALSE \
  INSTALL_EXTENSIONS=FALSE \
  ./start_linux.sh

# Parameters
model_url = "https://huggingface.co/andro-flock/LUSTIFY-SDXL-NSFW-checkpoint-v2-0-INPAINTING/resolve/main/README.md" #@param {type:"string"}
branch = "" #@param {type:"string"}
command_line_flags = "--load-in-4bit --use_double_quant" #@param {type:"string"}
api = True #@param {type:"boolean"}

if api:
  for param in ['--api', '--public-api']:
    if param not in command_line_flags:
      command_line_flags += f" {param}"

model_url = model_url.strip()
model_name = ""
if model_url != "":
    # Ensure the models directory exists
    !mkdir -p models

    initial_model_name_derived = "" # Store the base name before any wget suffix
    if not model_url.startswith('http'):
        model_url = 'https://huggingface.co/' + model_url

    branch = branch.strip()
    if '/resolve/' in model_url:
        initial_model_name_derived = model_url.split('?')[0].split('/')[-1]
        print(f"Downloading model {initial_model_name_derived} from {model_url}...")
        !wget -P models/ "{model_url}"
    else:
        url_parts = model_url.strip('/').split('/')
        initial_model_name_derived = f"{url_parts[-2]}_{url_parts[-1]}"
        print(f"Downloading model {initial_model_name_derived} from {model_url}...")
        if branch not in ['', 'main']:
            initial_model_name_derived += f"_{branch}"
            !wget -P models/ "{model_url}/resolve/{branch}/{initial_model_name_derived}"
        else:
            !wget -P models/ "{model_url}/resolve/main/{initial_model_name_derived}"

    # Determine the actual model_name after wget has potentially added suffixes
    # This loop will check for files starting with the expected name
    # and pick the first one found, assuming wget adds .1, .2 etc.
    found_model_file = None
    for f in Path('models').iterdir():
        if f.is_file() and f.name.startswith(initial_model_name_derived):
            found_model_file = f.name
            break # Pick the first match, assuming it's the one wget created

    if found_model_file:
        model_name = found_model_file
        print(f"Identified actual downloaded model file: {model_name}")
    else:
        # Fallback if no file is found (though wget should have created one)
        model_name = initial_model_name_derived
        print(f"Warning: Could not find actual downloaded model file. Using derived name: {model_name}")

    # Verify download (using the potentially updated model_name)
    print(f"Verifying download in /content/textgen/models/:\n")
    !ls -l models/

# Start the web UI
cmd = f"./start_linux.sh {command_line_flags} --share"
# Re-add --model flag now that manual wget download is confirmed
if model_name != "":
    # Pass the relative path to the model within the textgen directory
    cmd += f" --model models/{model_name}"

!$cmd